# 07c — Consultas espaciales MT: resúmenes operativos

Tercer bloque corto: métricas agregadas y conteos operativos.

> Esta notebook es una parte del bloque 07. Está separada para que sea más liviana de editar, ejecutar y revisar en clase.

## 1. Objetivo y prerequisitos

- Tener levantado PostGIS con `docker compose up -d postgis`.
- Haber ejecutado la notebook 06 para publicar `gis.mt_cables`, `gis.mt_postes` y `gis.mt_seccionadores`.
- Ejecutar las celdas en orden. Cada consulta deja el SQL visible antes del resultado.

## 2. Preparación de conexión

In [ ]:
# pathlib permite ubicar la raíz del proyecto aunque Jupyter se abra desde otra carpeta.
from pathlib import Path

# os permite leer variables de entorno para parametrizar la conexión local a PostGIS.
import os

# pandas se usa solo para mostrar resultados tabulares de forma cómoda en Jupyter.
import pandas as pd

# psycopg es el driver moderno de PostgreSQL usado por el proyecto.
try:
    import psycopg
    PSYCOPG_DISPONIBLE = True
except ModuleNotFoundError:
    psycopg = None
    PSYCOPG_DISPONIBLE = False
    print('No está instalado el paquete psycopg en este kernel.')
    print('Ejecutar desde la raíz del proyecto: python3 -m pip install -r requirements.txt')
    print('Después de instalar, reiniciar el kernel de Jupyter y volver a ejecutar la notebook.')

# Buscamos la raíz del proyecto subiendo hasta encontrar archivos propios del repo.
RAIZ = Path.cwd()
while RAIZ.parent != RAIZ:
    if (RAIZ / 'docker-compose.yml').exists() and (RAIZ / 'scripts' / 'gis_sqlserver.sh').exists():
        break
    RAIZ = RAIZ.parent

# Cargamos .env local sin pisar variables ya definidas por el entorno.
def cargar_env_local() -> None:
    env_path = RAIZ / '.env'
    if not env_path.exists():
        return
    for line in env_path.read_text(encoding='utf-8').splitlines():
        line = line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        key, value = line.split('=', 1)
        os.environ.setdefault(key.strip(), value.strip())

cargar_env_local()

DB_CONFIG = {
    'host': os.getenv('POSTGRES_HOST', 'localhost'),
    'port': int(os.getenv('POSTGRES_PORT', '5432')),
    'dbname': os.getenv('POSTGRES_DB', 'ceml_gis'),
    'user': os.getenv('POSTGRES_USER', 'ceml'),
    'password': os.getenv('POSTGRES_PASSWORD', 'ceml_admin_2026'),
}

SRID_PUBLICACION = 32721

# Esta función corta con una instrucción clara si falta el driver de PostgreSQL.
def exigir_psycopg() -> None:
    if not PSYCOPG_DISPONIBLE:
        raise RuntimeError(
            'Falta instalar psycopg en este kernel. '
            'Ejecutar: python -m pip install -r requirements.txt, '
            'reiniciar el kernel y volver a correr la notebook.'
        )

# Ejecuta una consulta SELECT y devuelve un DataFrame.
# El autocommit evita dejar transacciones abiertas durante ejercicios de solo lectura.
def ejecutar_sql(sql: str) -> pd.DataFrame:
    exigir_psycopg()
    with psycopg.connect(**DB_CONFIG, autocommit=True) as conn:
        with conn.cursor() as cur:
            cur.execute(sql)
            columnas = [col.name for col in cur.description]
            filas = cur.fetchall()
    return pd.DataFrame(filas, columns=columnas)

print('Raíz del proyecto:', RAIZ)
print('Base local:', DB_CONFIG['dbname'])
print('Host local:', DB_CONFIG['host'])
print('Puerto local:', DB_CONFIG['port'])
print('Usuario local:', DB_CONFIG['user'])
print('SRID de publicación:', SRID_PUBLICACION)

## 3. Verificación de capas GIS

In [ ]:
# Verificamos que estén publicadas las capas finales que usan estas consultas.
SQL_VERIFICAR_CAPAS = """
SELECT
    table_schema,
    table_name
FROM information_schema.tables
WHERE table_schema = 'gis'
  AND table_name IN ('mt_cables', 'mt_postes', 'mt_seccionadores')
ORDER BY table_name;
"""

ejecutar_sql(SQL_VERIFICAR_CAPAS)

## 4. Consulta 7: Longitud total de cables por cooperativa

**Consulta espacial:** Medición de geometrías ST_Length  
**Función:** Calcular longitud de red.  
**Consigna:** Calcular la cantidad de tramos y la longitud total de cables por cooperativa.

**Idea clave:** Como geom está en SRID 32721, ST_Length devuelve la longitud en metros. La consulta resume la red por cooperativa.

In [ ]:
SQL_CONSULTA_07 = """
SELECT
    c.coop,
    count(*) AS cantidad_tramos,
    round(sum(ST_Length(c.geom))::numeric, 2) AS longitud_total_m
FROM gis.mt_cables c
GROUP BY c.coop
ORDER BY longitud_total_m DESC;
"""

print(SQL_CONSULTA_07)

In [ ]:
ejecutar_sql(SQL_CONSULTA_07)

## 5. Consulta 8: Postes dentro del radio operativo de seccionadores

**Consulta espacial:** Proximidad entre capas ST_DWithin  
**Función:** Contar postes cercanos a seccionadores.  
**Consigna:** Para cada seccionador, contar cuántos postes hay dentro de un radio de 100 metros.

**Idea clave:** Relaciona seccionadores con postes cercanos. El LEFT JOIN permite conservar seccionadores aunque no tengan postes dentro del radio definido.

In [ ]:
SQL_CONSULTA_08 = """
SELECT
    s.id,
    s.idcad,
    s.coop,
    count(p.id) AS postes_en_radio
FROM gis.mt_seccionadores s
LEFT JOIN gis.mt_postes p
    ON ST_DWithin(p.geom, s.geom, 100)
GROUP BY s.id, s.idcad, s.coop
ORDER BY postes_en_radio DESC;
"""

print(SQL_CONSULTA_08)

In [ ]:
ejecutar_sql(SQL_CONSULTA_08)

## Cierre

Si este bloque ejecutó correctamente, continuá con la siguiente parte 07. La separación evita notebooks largas y facilita retomar desde el punto exacto donde quedó la clase.